# 8. Semantic web and RAG

Two more projections of the same model, both aimed at machine consumers:

| Surface | Module | Needs |
|---|---|---|
| RDF graph + SPARQL | `longeron.rdf` | the `rdf` extra (`pip install "longeron[rdf]"`) |
| retrieval chunks, neighborhoods, keyword search | `longeron.rag` | nothing — stdlib only |

**You will learn how to:**

- project a model onto RDF and inspect the triples;
- ask SPARQL questions over structure, requirements, and variation
  points (`rdf.sparql`);
- chunk a model into deterministic, re-parseable retrieval fragments
  (`rag.model_chunks`);
- walk semantic neighborhoods and run keyword search
  (`rag.neighborhood`, `rag.search`);
- wire the retrieve → cite → resolve agent loop through the
  interpreter.

**Prerequisites:** the `rdf` extra for the SPARQL half; `longeron.rag`
needs no extra. The examples reuse the UAV catalog from tutorial 7.

The RDF vocabulary is not invented here: `rdf:type` reuses the spec
metaclass per element that `longeron.api` emits as `@type` in Systems
Modeling API records (`part def` → `sysml:PartDefinition`), and property
names follow the spec's derived-property vocabulary (`ownedMember`,
`specializes`, `definedBy`, ...). The namespace is this package's *own*
(`https://sanbales.github.io/longeron/rdf/sysml#`): OMG has not
published an official RDF vocabulary for SysML v2, and squatting on a
plausible-looking OMG IRI would be worse than owning an honest one.
When an official vocabulary lands, the local names here map 1:1 (the
`longeron.rdf` module docs carry the fine print).

In [ ]:
from pathlib import Path

import rdflib

import longeron
from longeron import rdf

model = longeron.load(Path("../examples/uav_missions.sysml"), cache=False)
graph = rdf.to_graph(model)
print(f"{sum(1 for _ in model.iter_tree())} model elements -> {len(graph)} triples")

Element IRIs are minted from qualified names (anonymous elements fall
back to blank nodes), attribute *values* land as typed literals, and
doc comments become `rdfs:comment`. Here is one attribute and one
variation point, in Turtle:

In [ ]:
SYSML = rdflib.Namespace(rdf.VOCABULARY)


def excerpt(*qnames):
    sub = rdflib.Graph()
    sub.bind("sysml", SYSML)
    for qname in qnames:
        node = rdflib.URIRef(rdf.ELEMENT_BASE + qname.replace("::", "/"))
        for triple in graph.triples((node, None, None)):
            sub.add(triple)
    return sub.serialize(format="turtle")


print(excerpt("UavMissions::BoxQuad::mass", "UavMissions::Catalog::AirframeChoice"))

## Three SPARQL questions grep can't answer

`rdf.sparql` pre-binds the `sysml:`, `rdf:`, `rdfs:`, and `xsd:`
prefixes and accepts either a model or an already-built graph. First:
*which airframe families specialize `Airframe` and weigh under a
kilogram?* One pattern combines a specialization edge with a typed
literal comparison:

In [ ]:
rows = rdf.sparql(
    graph,
    """
    SELECT ?name ?mass WHERE {
        ?def a sysml:PartDefinition ; sysml:specializes ?super ;
             sysml:name ?name ; sysml:ownedMember ?attr .
        ?super sysml:name "Airframe" .
        ?attr sysml:name "mass" ; sysml:value ?mass .
        FILTER(?mass < 1.0)
    } ORDER BY ?mass
""",
)
for row in rows:
    print(f"{row.name}  {row.mass.toPython()} kg")

*Which requirements constrain which subject types?* The requirement
definitions declare their subjects, so traceability is a two-hop walk:

In [ ]:
rows = rdf.sparql(
    graph,
    """
    SELECT ?req ?subjectType WHERE {
        ?r a sysml:RequirementDefinition ; sysml:qualifiedName ?req ;
           sysml:ownedMember ?s .
        ?s sysml:kind "subject" ; sysml:definedBy ?t .
        ?t sysml:qualifiedName ?subjectType .
    } ORDER BY ?req
""",
)
for row in rows:
    print(f"{row.req}  ->  {row.subjectType}")

*What is the design space?* Every variation point and its variants,
straight off the `isVariation`/`isVariant` flags:

In [ ]:
rows = list(
    rdf.sparql(
        graph,
        """
    SELECT ?point ?variant ?target WHERE {
        ?p sysml:isVariation true ; sysml:name ?point ; sysml:ownedMember ?v .
        ?v sysml:isVariant true ; sysml:name ?variant ; sysml:definedBy ?t .
        ?t sysml:name ?target .
    } ORDER BY ?point ?variant
""",
    )
)
print(len(rows), "variants across the catalog:")
for row in rows[:6]:
    print(f"  {row.point} :: {row.variant} : {row.target}")
print("  ...")

Turtle and JSON-LD serialize with `rdf.to_turtle(model, path)` /
`rdf.to_jsonld(model, path)` for downstream triple stores.

## The retrieval substrate

`longeron.rag` is the other half: deterministic, dependency-free chunks
for embedding indexes and agent context windows.

- **ids are qualified names**, and chunking is deterministic. The same
  model yields byte-identical chunks, so embedding caches keyed on
  `(id, text)` stay warm across runs;
- **text is the exporter's own fragment printing**, so every chunk
  re-parses as SysML v2 (packages chunk *shallow*: their definitions
  are chunks of their own, so nothing is duplicated);
- **refs are outgoing qualified names**, canonicalized through the
  resolver, ready to be edges in a retrieval graph.

In [ ]:
from longeron import rag

chunks = rag.model_chunks(model)
print(len(chunks), "chunks")

chunk = next(c for c in chunks if c["id"] == "UavMissions::Catalog::AirframeChoice")
print("context:", chunk["context"])
print("refs:   ", ", ".join(chunk["refs"]))
print(chunk["text"])
longeron.loads(chunk["text"])  # every chunk re-parses
print("chunk text re-parses cleanly")

`neighborhood` is the graph-RAG helper: the chunks one (or *n*)
semantic hops from an element. Hops follow its types, its
specializations, the calcs it invokes, and (via reverse edges) whoever
references it. The shared `MissionUAV` assembly pulls its component
catalogs, its structural-sizing calcs, and the three missions that
specialize it:

In [ ]:
for c in rag.neighborhood(model, "UavMissions::MissionUAV", hops=1):
    print(f"{c['kind']:9s} {c['id']}")

`search` is the embedding-free fallback: TF-IDF-style token scoring,
camelCase-aware (`station` matches `stationMinutes`), stdlib only. The
substrate is useful before any embedding model enters the picture:

In [ ]:
for hit in rag.search(model, "station time energy", limit=5):
    print(f"{hit['score']:6.2f}  {hit['chunk']['id']}")

## How an agent consumes this

The point of both surfaces is *tool use*, not text retrieval. Generated
prose about a model is cheap; this package can **execute** the model,
so the reliable loop is:

1. **retrieve** — `rag.search` / `rag.neighborhood` (or a SPARQL query)
   gets the right elements into the context window;
2. **cite qualified names** — chunk ids *are* qualified names, so the
   agent's claims stay addressable instead of paraphrased;
3. **resolve for ground truth** — feed those names back through the
   `Interpreter`: evaluate the calc, instantiate the part, check the
   requirement. The model answers; the LLM only *routes*.

Retrieval found `StationTime`; the interpreter says what the ISR bird
actually delivers:

In [ ]:
hit = rag.search(model, "time on station minutes", limit=1)[0]["chunk"]
print("retrieved:", hit["id"])

interp = longeron.Interpreter(model)
prime = interp.instantiate("UavMissions::IsrPrime")
print("ground truth: stationMinutes =", round(prime.slots["stationMinutes"], 1))

result = interp.check_requirement("UavMissions::IsrStation", subject=prime)
print("IsrStation.stationFloor (>= 90 min):", "PASS" if result.requirements[0].passed else "FAIL")

An agent wired this way treats `longeron` as its *backend*: the
RDF/SPARQL view answers structural questions exactly, the chunks feed
whatever retrieval stack sits in front, and every number quoted to the
user comes from the interpreter, not from the language model's
imagination.